# Phase 3 — cGAN Colorization

**Task:** Same as Phase 1/2 — predict ab channels from the L channel.  
**Architecture:** ResNet-UNet generator (initialized from Phase 2) + PatchGAN discriminator.  
**Loss:** L1 + perceptual (keeps spatial structure) + adversarial LSGAN (pushes vivid colors).  
**Dependency:** Requires a Phase 2 checkpoint — set `PHASE2_CHECKPOINT` below before running.

---

## Project Overview — 3-Phase Colorization Pipeline

This project trains a deep learning model to colorize grayscale images in three progressively more powerful phases. Each phase produces increasingly realistic colors.

| Phase | Notebook | Architecture | Loss | Dependency |
|---|---|---|---|---|
| **1** | `01_unet.ipynb` | U-Net from scratch | L1 | None — fully independent baseline |
| **2** | `02_resnet_unet.ipynb` | ResNet-34 encoder + U-Net decoder | L1 + Perceptual | None — independent (uses ImageNet weights) |
| **3 (this)** | `03_cgan.ipynb` | Phase 2 generator + PatchGAN discriminator | L1 + Perceptual + Adversarial | Requires Phase 2 `best.pth` |

**Phase 1 — Baseline:** Trains a vanilla U-Net entirely from scratch. No pretrained weights, pure L1 pixel loss. The model will correctly learn *where* colors go (sky is blue, grass is green) but L1 loss minimizes average error, which causes the network to predict the safe mean color rather than committing to a vivid one. Expect correct spatial structure but washed-out, desaturated outputs. This phase exists as the reference point to measure how much each subsequent phase improves things.

**Phase 2 — Transfer Learning:** Swaps the encoder for a ResNet-34 pretrained on ImageNet. The frozen encoder supplies rich semantic features (textures, object boundaries) from epoch 1, so the decoder starts from a much stronger base. Adding a perceptual loss (VGG feature matching) pushes the model to match high-level structure, not just pixel values. Colors become more saturated and better localized, and convergence is 3–5× faster.

**Phase 3 — cGAN (this notebook):** Adds a PatchGAN discriminator that evaluates whether each 70×70 patch of the colorized image looks real. The adversarial loss eliminates the muddy, averaged colors that L1 alone produces — the generator is now penalized for *any* locally unconvincing output, pushing it toward vivid, photorealistic colorization. The generator is warm-started from Phase 2 weights so the discriminator has something meaningful to critique from epoch 1, keeping GAN training stable.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

from src.data.dataset import ColorizationDataset, lab_to_rgb
from src.models.resnet_unet import ResNetUNet
from src.models.discriminator import PatchGANDiscriminator
from src.losses.perceptual import PerceptualLoss

print(f"PyTorch {torch.__version__}")
DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Device: {DEVICE}")

## 2. Configuration

**Set `PHASE2_CHECKPOINT` to your Phase 2 best checkpoint before running.**

In [ ]:
PHASE2_CHECKPOINT = PROJECT_ROOT / "checkpoints" / "resnet_unet" / "best.pth"

CFG = {
    # Data
    "processed_dir": PROJECT_ROOT / "data" / "processed",
    "batch_size": 16,       # halved vs Phase 2 — two models in memory
    "num_workers": 4,
    "train_subset": 25000,       # set to None to train on the full split
    "subset_seed": 42,        # reproducible subset selection

    # Training
    "epochs": 30,
    "lr_g": 2e-4,           # generator lr
    "lr_d": 1e-4,           # discriminator lr — lower to avoid overwhelming G
    "betas": (0.5, 0.999),
    "lr_decay_start": 15,

    # Loss weights
    "lambda_l1": 1.0,
    "lambda_perceptual": 0.1,
    "lambda_adv": 0.01,     # adversarial term — small keeps training stable

    # Checkpointing
    "checkpoint_dir": PROJECT_ROOT / "checkpoints" / "cgan",
    "save_every": 5,
}

## 3. Dataset & DataLoader

In [ ]:
import pandas as pd

manifest_path = CFG["processed_dir"] / "manifest.csv"
_manifest = pd.read_csv(manifest_path)

_train_pool = _manifest[_manifest["split"] == "train"]
if CFG["train_subset"] is not None and CFG["train_subset"] < len(_train_pool):
    train_filenames = _train_pool.sample(
        n=CFG["train_subset"], random_state=CFG["subset_seed"]
    )["filename"].tolist()
    print(f"Training on subset of {len(train_filenames):,} images "
          f"(seed={CFG['subset_seed']}, full split has {len(_train_pool):,}).")
else:
    train_filenames = None
    print(f"Training on the full split of {len(_train_pool):,} images.")

train_ds = ColorizationDataset("train", processed_dir=CFG["processed_dir"],
                                horizontal_flip=True, filenames=train_filenames)
val_ds   = ColorizationDataset("val",   processed_dir=CFG["processed_dir"], horizontal_flip=False)
test_ds  = ColorizationDataset("test",  processed_dir=CFG["processed_dir"], horizontal_flip=False)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=CFG["num_workers"], pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=CFG["num_workers"], pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=CFG["num_workers"], pin_memory=True)

print(f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}  |  Test: {len(test_ds):,}")


## 4. Models

Generator is warm-started from Phase 2. Discriminator is randomly initialized.

In [ ]:
# Generator — load Phase 2 weights
generator = ResNetUNet(out_channels=2, freeze_encoder=False).to(DEVICE)
assert PHASE2_CHECKPOINT.exists(), f"Phase 2 checkpoint not found: {PHASE2_CHECKPOINT}"
generator.load_state_dict(torch.load(PHASE2_CHECKPOINT, map_location=DEVICE))
print(f"Generator loaded from {PHASE2_CHECKPOINT}")

# Discriminator — random init, takes concat(L, ab) = 3 channels
discriminator = PatchGANDiscriminator(in_channels=3).to(DEVICE)
print("Discriminator initialized (random weights)")

g_params = sum(p.numel() for p in generator.parameters())
d_params = sum(p.numel() for p in discriminator.parameters())
print(f"\nGenerator:     {g_params:,} params")
print(f"Discriminator: {d_params:,} params")

# Sanity check
with torch.no_grad():
    dummy_l  = torch.randn(2, 1, 256, 256, device=DEVICE)
    fake_ab  = generator(dummy_l)
    d_out    = discriminator(dummy_l, fake_ab)
    print(f"\nGenerator output:     {fake_ab.shape}  (expected: (2, 2, 256, 256))")
    print(f"Discriminator output: {d_out.shape}    (expected: (2, 1, 30, 30))")

## 5. Loss Functions

**LSGAN** (least-squares GAN) instead of vanilla BCE — smoother gradients, less prone to vanishing.

- D loss: `MSE(D(real), 1) + MSE(D(fake), 0)`  
- G adversarial loss: `MSE(D(fake), 1)` — generator wants discriminator to score its output as real

In [ ]:
l1_criterion         = nn.L1Loss()
perceptual_criterion = PerceptualLoss().to(DEVICE)
mse_criterion        = nn.MSELoss()   # LSGAN

def discriminator_loss(d_real: torch.Tensor, d_fake: torch.Tensor) -> torch.Tensor:
    """LSGAN discriminator loss: real→1, fake→0."""
    real_loss = mse_criterion(d_real, torch.ones_like(d_real))
    fake_loss = mse_criterion(d_fake, torch.zeros_like(d_fake))
    return (real_loss + fake_loss) * 0.5

def generator_adv_loss(d_fake: torch.Tensor) -> torch.Tensor:
    """LSGAN generator adversarial loss: fool discriminator → fake scores as real."""
    return mse_criterion(d_fake, torch.ones_like(d_fake))

print("Losses: L1, Perceptual (VGG-16), LSGAN adversarial")

## 6. Optimizers & Schedulers

In [ ]:
opt_g = torch.optim.Adam(generator.parameters(),     lr=CFG["lr_g"], betas=CFG["betas"])
opt_d = torch.optim.Adam(discriminator.parameters(), lr=CFG["lr_d"], betas=CFG["betas"])

t_max = CFG["epochs"] - CFG["lr_decay_start"]
sched_g = torch.optim.lr_scheduler.CosineAnnealingLR(opt_g, T_max=t_max, eta_min=1e-6)
sched_d = torch.optim.lr_scheduler.CosineAnnealingLR(opt_d, T_max=t_max, eta_min=1e-6)

## 7. Training

In [ ]:
def train_epoch(generator, discriminator, loader, opt_g, opt_d, device):
    generator.train()
    discriminator.train()

    totals = {"g": 0.0, "g_l1": 0.0, "g_perc": 0.0, "g_adv": 0.0, "d": 0.0}

    pbar = tqdm(loader, desc="  train", leave=False, unit="batch")
    for l_batch, ab_batch in pbar:
        l_batch  = l_batch.to(device)
        ab_batch = ab_batch.to(device)
        n        = l_batch.size(0)

        fake_ab = generator(l_batch)

        # Discriminator step
        opt_d.zero_grad()
        d_real = discriminator(l_batch, ab_batch)
        d_fake = discriminator(l_batch, fake_ab.detach())
        loss_d = discriminator_loss(d_real, d_fake)
        loss_d.backward()
        opt_d.step()

        # Generator step
        opt_g.zero_grad()
        d_fake_for_g = discriminator(l_batch, fake_ab)
        loss_l1   = l1_criterion(fake_ab, ab_batch)
        loss_perc = perceptual_criterion(l_batch, fake_ab, l_batch, ab_batch)
        loss_adv  = generator_adv_loss(d_fake_for_g)
        loss_g = (
            CFG["lambda_l1"]          * loss_l1
            + CFG["lambda_perceptual"] * loss_perc
            + CFG["lambda_adv"]        * loss_adv
        )
        loss_g.backward()
        opt_g.step()

        totals["g"]      += loss_g.item()    * n
        totals["g_l1"]   += loss_l1.item()   * n
        totals["g_perc"] += loss_perc.item() * n
        totals["g_adv"]  += loss_adv.item()  * n
        totals["d"]      += loss_d.item()    * n
        pbar.set_postfix(G=f"{loss_g.item():.4f}", D=f"{loss_d.item():.4f}")

    N = len(loader.dataset)
    return {k: v / N for k, v in totals.items()}


@torch.no_grad()
def eval_epoch(generator, loader, device):
    generator.eval()
    total_l1 = total_psnr = 0.0
    for l_batch, ab_batch in tqdm(loader, desc="  val  ", leave=False, unit="batch"):
        l_batch  = l_batch.to(device)
        ab_batch = ab_batch.to(device)
        pred_ab  = generator(l_batch)
        total_l1   += l1_criterion(pred_ab, ab_batch).item() * l_batch.size(0)
        total_psnr += psnr(pred_ab.cpu(), ab_batch.cpu())    * l_batch.size(0)
    N = len(loader.dataset)
    return total_l1 / N, total_psnr / N

In [ ]:
CFG["checkpoint_dir"].mkdir(parents=True, exist_ok=True)

steps_per_epoch = len(train_loader)
print(f"Phase 3 — cGAN (generator warm-started from Phase 2)")
print(f"  {CFG['epochs']} epochs x {steps_per_epoch:,} steps/epoch = {CFG['epochs'] * steps_per_epoch:,} total steps")
print(f"  Device: {DEVICE}  |  Batch size: {CFG['batch_size']}  |  lr_G: {CFG['lr_g']}  lr_D: {CFG['lr_d']}")
print(f"  Loss weights: L1={CFG['lambda_l1']}  Perceptual={CFG['lambda_perceptual']}  Adversarial={CFG['lambda_adv']}")
print(f"  Checkpoints -> {CFG['checkpoint_dir']}")
print(f"\n  Healthy training signals to watch:")
print(f"    D_loss ~ 0.25  (LSGAN equilibrium — too low means D is overpowering G)")
print(f"    G_adv decreasing  (G learning to fool D)")
print(f"    val_l1 stable  (close to Phase 2 final — a spike means GAN destabilized)\n")

history = {
    "g_loss": [], "g_l1": [], "g_perc": [], "g_adv": [],
    "d_loss": [], "val_l1": [], "val_psnr": [],
}
best_val_l1 = float("inf")

for epoch in range(1, CFG["epochs"] + 1):
    t0 = time.time()

    train_losses     = train_epoch(generator, discriminator, train_loader, opt_g, opt_d, DEVICE)
    val_l1, val_psnr = eval_epoch(generator, val_loader, DEVICE)

    if epoch >= CFG["lr_decay_start"]:
        sched_g.step()
        sched_d.step()

    history["g_loss"].append(train_losses["g"])
    history["g_l1"].append(train_losses["g_l1"])
    history["g_perc"].append(train_losses["g_perc"])
    history["g_adv"].append(train_losses["g_adv"])
    history["d_loss"].append(train_losses["d"])
    history["val_l1"].append(val_l1)
    history["val_psnr"].append(val_psnr)

    elapsed = time.time() - t0
    eta_s   = elapsed * (CFG["epochs"] - epoch)
    eta_str = f"{eta_s/3600:.1f}h" if eta_s > 3600 else f"{eta_s/60:.0f}min"

    print(
        f"[{epoch:3d}/{CFG['epochs']}]  "
        f"G={train_losses['g']:.4f}  "
        f"[l1={train_losses['g_l1']:.4f} perc={train_losses['g_perc']:.4f} adv={train_losses['g_adv']:.4f}]  "
        f"D={train_losses['d']:.4f}  "
        f"val_l1={val_l1:.4f}  PSNR={val_psnr:.2f}dB  "
        f"{elapsed:.0f}s/epoch  ETA {eta_str}"
    )

    if val_l1 < best_val_l1:
        best_val_l1 = val_l1
        torch.save(generator.state_dict(), CFG["checkpoint_dir"] / "best_generator.pth")
        print(f"         -> New best! val_l1={best_val_l1:.4f}  saved best_generator.pth")

    if epoch % CFG["save_every"] == 0:
        ckpt_name = f"epoch_{epoch:03d}.pth"
        torch.save(
            {
                "epoch": epoch,
                "generator":     generator.state_dict(),
                "discriminator": discriminator.state_dict(),
                "opt_g":         opt_g.state_dict(),
                "opt_d":         opt_d.state_dict(),
                "history":       history,
            },
            CFG["checkpoint_dir"] / ckpt_name,
        )
        print(f"         -> Checkpoint saved: {ckpt_name}")

print(f"\nTraining complete. Best val L1: {best_val_l1:.4f}")

## 8. Training Curves

**Healthy GAN signals to watch for:**
- `D_loss` converges around 0.25 (random chance for LSGAN) — too low means D is overwhelming G
- `G_adv` should gradually decrease as G learns to fool D
- `val_l1` should stay close to Phase 2's final val L1 — a big spike means GAN destabilized

In [ ]:
epochs_range = range(1, len(history["g_loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

axes[0].plot(epochs_range, history["g_loss"],  label="G total")
axes[0].plot(epochs_range, history["d_loss"],  label="D", linestyle="--")
axes[0].axhline(0.25, color="gray", linewidth=0.8, linestyle=":", label="D target (0.25)")
axes[0].set_title("Generator vs Discriminator Loss")
axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history["g_l1"],   label="L1")
axes[1].plot(epochs_range, history["g_perc"], label="Perceptual", linestyle="--")
axes[1].plot(epochs_range, history["g_adv"],  label="Adversarial", linestyle=":")
axes[1].set_title("Generator Loss Components")
axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_range, history["val_l1"],   label="Val L1", color="tab:blue")
ax2 = axes[2].twinx()
ax2.plot(epochs_range, history["val_psnr"], label="Val PSNR", color="tab:green", linestyle="--")
axes[2].set_title("Validation Metrics")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("L1", color="tab:blue")
ax2.set_ylabel("PSNR (dB)", color="tab:green")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Visualize Colorization Results

Each row: **Grayscale input → cGAN prediction → Ground truth**.

In [ ]:
generator.load_state_dict(torch.load(CFG["checkpoint_dir"] / "best_generator.pth", map_location=DEVICE))
generator.eval()
print("Loaded best generator checkpoint.")

In [ ]:
@torch.no_grad()
def visualize_predictions(model, loader, device, n=8, title="cGAN colorization"):
    l_batch, ab_batch = next(iter(loader))
    l_batch, ab_batch = l_batch[:n].to(device), ab_batch[:n].to(device)

    pred_ab  = model(l_batch).cpu()
    l_batch  = l_batch.cpu()
    ab_batch = ab_batch.cpu()

    fig, axes = plt.subplots(3, n, figsize=(2.5 * n, 8))
    fig.suptitle(title, fontsize=13)
    row_labels = ["Grayscale (input)", "Predicted (cGAN)", "Ground truth"]

    for i in range(n):
        gray      = (l_batch[i, 0].numpy() + 1.0) / 2.0
        pred_rgb  = lab_to_rgb(l_batch[i], pred_ab[i])
        truth_rgb = lab_to_rgb(l_batch[i], ab_batch[i])

        axes[0, i].imshow(gray, cmap="gray", vmin=0, vmax=1)
        axes[1, i].imshow(pred_rgb)
        axes[2, i].imshow(truth_rgb)

        for row in range(3):
            axes[row, i].axis("off")
            if i == 0:
                axes[row, i].set_ylabel(row_labels[row], fontsize=9)

    plt.tight_layout()
    plt.show()

visualize_predictions(generator, val_loader, DEVICE)

## 10. Test-set Evaluation

In [ ]:
test_l1, test_psnr = eval_epoch(generator, test_loader, DEVICE)
print(f"Test L1  : {test_l1:.4f}")
print(f"Test PSNR: {test_psnr:.2f} dB")

## 11. Checkpoint Info

In [ ]:
# Resume full GAN training from a periodic checkpoint:
#
# ckpt = torch.load(CFG["checkpoint_dir"] / "epoch_015.pth", map_location=DEVICE)
# generator.load_state_dict(ckpt["generator"])
# discriminator.load_state_dict(ckpt["discriminator"])
# opt_g.load_state_dict(ckpt["opt_g"])
# opt_d.load_state_dict(ckpt["opt_d"])
# history    = ckpt["history"]
# start_epoch = ckpt["epoch"] + 1
print("Checkpoints saved to:", CFG["checkpoint_dir"])
print("  best_generator.pth — generator with lowest val L1")
print("  epoch_XXX.pth      — full state (G + D + optimizers + history) every", CFG["save_every"], "epochs")